# NB07s — Avaluació Exhaustiva d'Agents (ENTORN SPARSE)

Notebook per avaluar **agents entrenats en l'entorn sparse** sobre el split de test.

## 4 Experiments d'Avaluació

| Experiment | Mode Trading | Rolling Windows | Full Period |
|----------|--------------|-----------------|-------------|
| **Ex.1** | All-in All-out | 4w, 8w, 12w | — |
| **Ex.2** | Position Sizing (10% max) | 4w, 8w, 12w | — |
| **Ex.3** | All-in All-out | — | ✓ (reajust setmanal meta) |
| **Ex.4** | Position Sizing | — | ✓ (reajust setmanal meta) |

## Configuració Agent

**ÚNIC PARÀMETRE A CANVIAR**: CEL·LA 2

- `AGENT_TYPE`: nom del model
- `META_ADJUSTMENT`: None | "Fixed" | "Loss"

## Sortides

`results_sparse/10s_evaluation/{AGENT_NAME}/`
- 10 JSON (config + experiments + resum)
- 20 PNG (visualitzacions)
- 1 CSV (emissions CO₂)


In [ ]:
"""
NB10s — Avaluació Exhaustiva d'Agents (ENTORN SPARSE)

Script temporal de desenvolupament. Després es convertirà a .ipynb

4 Experiments:
  Ex.1: Rolling windows 4w/8w/12w all-in
  Ex.2: Rolling windows 4w/8w/12w position sizing
  Ex.3: Full period all-in amb reajustos setmanals (meta)
  Ex.4: Full period sizing amb reajustos setmanals (meta)
"""

## IMPORTS I CONFIGURACIÓ


In [ ]:
import json
import os as _os
import random
import time
import warnings
import copy
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# CodeCarbon per tracking emissions
try:
    from codecarbon import EmissionsTracker
    HAS_CODECARBON = True
except ImportError:
    print("[WARNING] codecarbon no instal·lat. Emissions no es trackejaran.")
    HAS_CODECARBON = False

from src.data.manager import DataManager
from src.rl.envs.crypto_market_env_consolidated_reward import (
    CryptoMarketEnvConsolidatedReward as CryptoMarketEnv,
    INITIAL_BALANCE, MAX_EPISODE_STEPS,
    HOLD_PENALTY, steps_per_4h_for,
)
from src.rl.metrics import compute_sharpe, compute_max_drawdown
from src.rl.agents.actor_critic_policy import ActorCriticPolicy
from src.rl.agents.feudal_agent import FeudalAgent
from src.rl.agents.feudal_agent_paper import FeUdalAgent
from src.rl.agents.feudal_agent_v0 import FeudalAgentV0
from src.rl.agents.feudal_agent_v1 import FeudalAgentV1
from src.rl.agents.feudal_agent_v2 import FeudalAgentV2
from src.rl.agents.feudal_agent_v3 import FeudalAgentV3
from src.rl.agents.feudal_agent_v4 import FeudalAgentV4
from src.rl.agents.feudal_agent_v5 import FeudalAgentV5
from src.rl.agents.feudal_agent_v6 import FeudalAgentV6
from src.rl.agents.feudal_agent_v7 import FeudalAgentV7
from src.rl.agents.feudal_agent_v8 import FeudalAgentV8
from src.rl.agents.feudal_agent_v9 import FeudalAgentV9

warnings.filterwarnings("ignore")

# ── Reproducibilitat ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Working directory ─────────────────────────────────────────────────────────
_cwd = Path.cwd()
if not (_cwd / "data" / "processed").exists():
    _root = _cwd.parents[1]
    if (_root / "data" / "processed").exists():
        _os.chdir(_root)
        print(f"CWD -> {Path.cwd()}")

# ── Constants ─────────────────────────────────────────────────────────────────
TIMEFRAME = "60m"
STEPS_PER_YEAR = 365 * 24  # 8760 per interval 60m
STEPS_PER_WEEK = 7 * 24    # 168 steps @ 60m

ACTION_NAMES = {0: "HOLD", 1: "LONG", 2: "SHORT", 3: "CLOSE"}

## CONFIGURACIÓ AGENT (ÚNIC PARÀMETRE A CANVIAR)


In [ ]:
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  MODIFICAR AQUÍ                                                          │
# └─────────────────────────────────────────────────────────────────────────┘

AGENT_TYPE = "hrl_feudal_v5"
META_ADJUSTMENT = None  # None | "Fixed" | "Loss"

# Paràmetres meta-adaptation
META_ADAPT_STEPS = 1_000
META_ADAPT_LOSS_THRESHOLD = 0.0005
META_ADAPT_MAX_STEPS = 10_000
META_ADAPT_WEEKS = 6

# Agents disponibles
AVAILABLE_AGENTS = {
    "ppo_pytorch_sparse_final": {
        "path": "checkpoints_sparse/ppo_pytorch_sparse_final.pt",
        "type": "base",
        "loader": "ActorCriticPolicy",
    },
    "ppo_reptile_meta_adj": {
        "path": "checkpoints_sparse/ppo_reptile_meta_adj_adaptative.pt",
        "type": "meta_reptile",
        "loader": "ActorCriticPolicy",
    },
    "ppo_reptile_meta_new": {
        "path": "checkpoints_sparse/ppo_reptile_meta_new_adaptative.pt",
        "type": "meta_reptile",
        "loader": "ActorCriticPolicy",
    },
    "ppo_fomaml_meta_adj": {
        "path": "checkpoints_sparse/ppo_fomaml_meta_adj_adaptative.pt",
        "type": "meta_fomaml",
        "loader": "ActorCriticPolicy",
    },
    "ppo_fomaml_meta_new": {
        "path": "checkpoints_sparse/ppo_fomaml_meta_new_adaptative.pt",
        "type": "meta_fomaml",
        "loader": "ActorCriticPolicy",
    },
    # ── HRL FeUdal agents v0–v6 (sparse) ─────────────────────────────────
    "hrl_feudal_v0": {
        "path": "checkpoints_sparse/hrl_sparse_feudal_agent/hrl_feudal_best.pt",
        "type": "base",
        "loader": "FeudalAgentV0",
        "best_params_path": "results_sparse/05s_hrl_FeUdal_agent/hp_analysis/best_params.json",
    },
    "hrl_feudal_v1": {
        "path": "checkpoints_sparse/hrl_sparse_feudal_agent_relu/hrl_feudal_best.pt",
        "type": "base",
        "loader": "FeudalAgentV1",
        "best_params_path": "results_sparse/05s_hrl_FeUdal_agent_relu/hp_analysis/best_params.json",
    },
    "hrl_feudal_v2": {
        "path": "checkpoints_sparse/hrl_sparse_feudal_agent_rnn/hrl_feudal_best.pt",
        "type": "base",
        "loader": "FeudalAgentV2",
        "best_params_path": "results_sparse/05s_hrl_FeUdal_agent_rnn/hp_analysis/best_params.json",
    },
    "hrl_feudal_v3": {
        "path": "checkpoints_sparse/hrl_sparse_feudal_agent_relu_N/hrl_feudal_best.pt",
        "type": "base",
        "loader": "FeudalAgentV3",
        "best_params_path": "results_sparse/05s_hrl_FeUdal_agent_relu_N/hp_analysis/best_params.json",
    },
    "hrl_feudal_v4": {
        "path": "checkpoints_sparse/hrl_sparse_feudal_agent_paper/hrl_feudal_best.pt",
        "type": "base",
        "loader": "FeudalAgentV4",
        "best_params_path": "results_sparse/05s_hrl_FeUdal_agent_paper/hp_analysis/best_params.json",
    },
    "hrl_feudal_v5": {
        "path": "checkpoints_sparse/hrl_sparse_feudal_agent_paper_1phase/hrl_feudal_best.pt",
        "type": "base",
        "loader": "FeudalAgentV5",
        "best_params_path": "results_sparse/05s_hrl_FeUdal_agent_paper_1phase/hp_analysis/best_params.json",
    },
    "hrl_feudal_v6": {
        "path": "checkpoints_sparse/hrl_sparse_feudal_agent_paper_1ph_pt/hrl_feudal_best.pt",
        "type": "base",
        "loader": "FeudalAgentV6",
        "best_params_path": "results_sparse/05s_hrl_FeUdal_agent_paper_1ph_pt/hp_analysis/best_params.json",
    },
}
# Validació
if AGENT_TYPE not in AVAILABLE_AGENTS:
    raise ValueError(f"AGENT_TYPE desconegut: '{AGENT_TYPE}'")

agent_cfg = AVAILABLE_AGENTS[AGENT_TYPE]
IS_META_AGENT = agent_cfg["type"].startswith("meta_")

if META_ADJUSTMENT is not None and not IS_META_AGENT:
    raise ValueError(f"META_ADJUSTMENT només vàlid per agents meta")

if META_ADJUSTMENT not in (None, "Fixed", "Loss"):
    raise ValueError(f"META_ADJUSTMENT invàlid: {META_ADJUSTMENT}")

# Nom agent
AGENT_NAME = AGENT_TYPE if META_ADJUSTMENT is None else f"{AGENT_TYPE}_{META_ADJUSTMENT.lower()}"
# Etiqueta agent per títols de gràfics
import re as _re
_vn = _re.search(r'v(\d+)', AGENT_TYPE)
AGENT_LABEL = f"FeUdal Agent v{_vn.group(1)}" if _vn else AGENT_NAME
REWARD_TYPE = "Sparse"

# Directori resultats
RESULTS_DIR = Path("results_sparse") / "10s_evaluation" / AGENT_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for subdir in ["ex1_rolling_allin", "ex2_rolling_sizing", "ex3_full_allin", "ex4_full_sizing", "carbon"]:
    (RESULTS_DIR / subdir).mkdir(exist_ok=True)

print(RESULTS_DIR  )

## DADES


In [ ]:
dm = DataManager(interval=TIMEFRAME, project_root=Path("."))
FEATURE_COLS = dm.features
N_FEATURES = len(FEATURE_COLS)

df_train, df_val, df_test = dm.load_all()

print(f"Device:        {DEVICE}")
print(f"Features:      {N_FEATURES}")
print(f"Test:          {len(df_test):,} rows")
print(f"CodeCarbon:    {'✓' if HAS_CODECARBON else '✗'}")

## ENTORN


In [ ]:
def make_env(df: pl.DataFrame, **kwargs) -> CryptoMarketEnv:
    return CryptoMarketEnv(
        df=df,
        feature_cols=FEATURE_COLS,
        steps_per_4h=steps_per_4h_for(TIMEFRAME),
        **kwargs,
    )

## CLASSES I FUNCIONS


In [ ]:
# ── Wrapper HRL FeUdal Agent ──────────────────────────────────────────────────

class HRLAgentWrapper:
    """Wrapper per FeudalAgent que exposa la mateixa interfície predict() que ActorCriticPolicy.

    FeudalAgent.predict_step() necessita estat intern per episodi (step_ep, cur_goal).
    Aquest wrapper gestiona l'estat i ofereix reset_episode() per reiniciar-lo.
    """

    def __init__(self, feudal_agent: FeudalAgent, c: int, device):
        self.agent = feudal_agent
        self.c = c
        self.device = device
        self._step_ep = 0
        self._cur_goal = None

    def reset_episode(self):
        self._step_ep = 0
        self._cur_goal = None

    def predict(self, obs: np.ndarray, deterministic: bool = True):
        action, self._cur_goal, *_ = self.agent.predict_step(
            obs=obs,
            step_ep=self._step_ep,
            c=self.c,
            cur_goal=self._cur_goal,
            device=self.device,
            deterministic=deterministic,
        )
        self._step_ep += 1
        return action, None

    def count_params(self) -> int:
        return sum(p.numel() for p in self.agent.parameters())


# ── Wrapper Meta-Agent ────────────────────────────────────────────────────────

class MetaAgentWrapper:
    """Wrapper per agents meta amb capacitat d'adaptació fast-learning."""

    def __init__(self, base_policy: ActorCriticPolicy, meta_cfg: dict, device):
        self.base_policy = base_policy
        self.adapted = None
        self.meta_cfg = meta_cfg
        self.device = device
        self.adaptation_log = []

    def adapt(
        self,
        df_support: pl.DataFrame,
        mode: str,
        n_steps: int = 5_000,
        loss_threshold: float = 0.01,
        max_steps: int = 10_000,
    ) -> Dict[str, Any]:
        """Adapta política sobre df_support. Mode: Fixed | Loss"""

        self.adapted = copy.deepcopy(self.base_policy).to(self.device)
        self.adapted.train()

        # CodeCarbon tracker
        tracker = None
        if HAS_CODECARBON:
            tracker = EmissionsTracker(
                project_name=f"meta_adapt_{AGENT_NAME}",
                output_dir=str(RESULTS_DIR / "carbon"),
                log_level="error",
                save_to_file=True,
            )
            tracker.start()

        t_start = time.time()

        lr = self.meta_cfg.get("learning_rate", 3e-4)
        optimizer = torch.optim.Adam(self.adapted.parameters(), lr=lr)

        env_adapt = make_env(df_support, max_episode_steps=len(df_support)-1)

        target_steps = n_steps if mode == "Fixed" else max_steps
        steps_done = 0
        final_loss = float("inf")

        pbar = tqdm(total=target_steps, desc=f"Meta-adapt ({mode})", leave=False)

        while steps_done < target_steps:
            obs, info = env_adapt.reset()
            done = False
            ep_loss = []

            while not done and steps_done < target_steps:
                action, _ = self.adapted.predict(obs, deterministic=False)
                obs_next, reward, term, trunc, info = env_adapt.step(int(action))
                done = term or trunc

                obs_t = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
                action_t = torch.as_tensor(action, dtype=torch.long, device=self.device)
                reward_t = torch.as_tensor(reward, dtype=torch.float32, device=self.device)

                logits, value = self.adapted.forward(obs_t.unsqueeze(0))
                dist = torch.distributions.Categorical(logits=logits)
                log_prob = dist.log_prob(action_t)

                loss = -(log_prob * reward_t).mean()

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.adapted.parameters(), 0.5)
                optimizer.step()

                ep_loss.append(loss.item())
                steps_done += 1
                pbar.update(1)

                obs = obs_next

                if mode == "Loss" and steps_done % 100 == 0:
                    recent_loss = np.mean(ep_loss[-100:]) if len(ep_loss) >= 100 else np.mean(ep_loss)
                    if np.absolute(recent_loss) < loss_threshold:
                        print(f"  [Convergència: loss={recent_loss:.4f}]")
                        break

            final_loss = np.mean(ep_loss) if ep_loss else final_loss

            if mode == "Loss" and np.absolute(final_loss) < loss_threshold:
                break

        pbar.close()

        t_elapsed = time.time() - t_start
        co2_kg = tracker.stop() if tracker else 0.0

        self.adapted.eval()

        result = {
            "time_sec": round(t_elapsed, 2),
            "gradient_steps": steps_done,
            "final_loss": round(final_loss, 6),
            "co2_kg": round(co2_kg, 8) if co2_kg else 0.0,
            "mode": mode,
        }

        self.adaptation_log.append(result)

        print(f"  Adaptació {mode}: {steps_done} steps, loss={final_loss:.4f}, "
              f"time={t_elapsed:.1f}s, CO2={co2_kg:.6f}kg")

        return result

    def predict(self, obs: np.ndarray, deterministic: bool = True):
        model = self.adapted if self.adapted is not None else self.base_policy
        return model.predict(obs, deterministic=deterministic)

    def reset_adaptation(self):
        self.adapted = None


# ── Càrrega agent ─────────────────────────────────────────────────────────────

# ── Mapa loader → classe agent HRL ──────────────────────────────────────────
_HRL_CLASSES = {
    "FeudalAgent":   FeudalAgent,
    "FeUdalAgent":   FeUdalAgent,
    "FeudalAgentV0": FeudalAgentV0,
    "FeudalAgentV1": FeudalAgentV1,
    "FeudalAgentV2": FeudalAgentV2,
    "FeudalAgentV3": FeudalAgentV3,
    "FeudalAgentV4": FeudalAgentV4,
    "FeudalAgentV5": FeudalAgentV5,
    "FeudalAgentV6": FeudalAgentV6,
    "FeudalAgentV7": FeudalAgentV7,
    "FeudalAgentV8": FeudalAgentV8,
    "FeudalAgentV9": FeudalAgentV9,
}


def _load_agent_with_meta(
    agent_path: str,
    agent_type: str,
    meta_adjustment: Optional[str],
    device: torch.device,
    loader: str = "ActorCriticPolicy",
    c: int = 0,
    best_params_path: Optional[str] = None,
) -> Tuple[Any, bool]:
    """Carrega agent i retorna (model, is_meta_wrapper).

    Per agents HRL, c s'extreu automàticament del checkpoint (config["c"] o
    hp["time_horizon"]). Si no hi és, es busca a best_params_path. Finalment
    s'usa el paràmetre c (0 = no especificat).
    """

    if not Path(agent_path).exists():
        raise FileNotFoundError(f"Checkpoint no trobat: {agent_path}")

    print(f"Carregant: {agent_path}")

    # ── Qualsevol agent HRL FeUdal (v0–v9 + llegats) ──────────────────────
    if loader in _HRL_CLASSES:
        ckpt_raw = torch.load(str(agent_path), map_location=device, weights_only=False)

        # Extreu c: 1) config["c"]  2) hp["time_horizon"] (v9)  3) best_params.json  4) c param
        effective_c = None
        if "config" in ckpt_raw and "c" in (ckpt_raw.get("config") or {}):
            effective_c = int(ckpt_raw["config"]["c"])
        elif "hp" in ckpt_raw and "time_horizon" in (ckpt_raw.get("hp") or {}):
            effective_c = int(ckpt_raw["hp"]["time_horizon"])
        elif best_params_path and Path(best_params_path).exists():
            bp = json.loads(Path(best_params_path).read_text(encoding="utf-8"))
            effective_c = int(bp["best_params"]["c"])
        elif c > 0:
            effective_c = c
        else:
            raise ValueError(
                f"No s'ha pogut determinar 'c' per {agent_path}. "
                "Afegeix best_params_path o c a AVAILABLE_AGENTS."
            )

        cls = _HRL_CLASSES[loader]
        agent = cls.load(str(agent_path), device=device)
        n_params = sum(p.numel() for p in agent.parameters())
        print(f"  Paràmetres: {n_params:,}")
        print(f"  Mode: HRL FeUdal {loader} (c={effective_c})")
        wrapper = HRLAgentWrapper(agent, c=effective_c, device=device)
        return wrapper, False

    # ── ActorCriticPolicy (PPO base / meta) ───────────────────────────────
    policy = ActorCriticPolicy.load(agent_path, device=device)
    n_params = policy.count_params()
    print(f"  Paràmetres: {n_params:,}")

    is_meta = agent_type.startswith("meta_")
    if not is_meta or meta_adjustment is None:
        print(f"  Mode: {'Meta sense ajustos' if is_meta else 'Base'}")
        return policy, False

    print(f"  Mode: Meta amb ajustos ({meta_adjustment})")

    ckpt = torch.load(agent_path, map_location=device, weights_only=False)
    meta_cfg = ckpt.get("hyperparams", {})

    wrapper = MetaAgentWrapper(policy, meta_cfg, device)

    return wrapper, True

# ── Action masking ────────────────────────────────────────────────────────────

def _correct_action(raw_action: int, pos_side: int) -> int:
    """Corregeix acció si no és vàlida."""
    if pos_side == 0:
        if raw_action == 3:
            return 0
    else:
        if raw_action in (1, 2):
            return 0
    return raw_action


# ── Reconstrucció posicions ───────────────────────────────────────────────────

def _reconstruct_positions(actions: List[int]) -> List[int]:
    """Reconstrueix seqüència posicions: 0=FLAT, 1=LONG, -1=SHORT."""
    pos = 0
    positions = []
    for a in actions:
        if a == 1: pos = 1
        elif a == 2: pos = -1
        elif a == 3: pos = 0
        positions.append(pos)
    return positions


# ── Càlcul mètriques ──────────────────────────────────────────────────────────

def _calculate_metrics(
    equities: List[float],
    actions: List[int],
    prices: List[float],
    n_trades: int,
    fees_paid: float,
    n_corrections: int = 0,
    steps_per_year: int = STEPS_PER_YEAR,
) -> Dict[str, Any]:
    """Calcula les 14 mètriques obligatòries."""

    eq = np.array(equities, dtype=np.float64)
    n_steps = len(actions)

    # Financeres
    total_return = float((eq[-1] / eq[0] - 1) * 100) if len(eq) > 1 else 0.0
    sharpe = compute_sharpe(equities, steps_per_year=steps_per_year)
    max_dd = compute_max_drawdown(equities)
    bah = float((prices[-1] / prices[0] - 1) * 100) if prices else 0.0
    alpha = total_return - bah

    # Trading
    n_weeks = n_steps / STEPS_PER_WEEK
    trade_freq = n_trades / n_weeks if n_weeks > 0 else 0.0

    eq_diffs = np.diff(eq)
    wins = np.sum(eq_diffs > 0)
    win_rate = (wins / len(eq_diffs) * 100) if len(eq_diffs) > 0 else 0.0

    positions = _reconstruct_positions(actions)
    pos_arr = np.array(positions)

    changes = [0] + [i for i in range(1, len(pos_arr)) if pos_arr[i] != pos_arr[i-1]]
    durations = [changes[i+1] - changes[i] for i in range(len(changes)-1)]
    if len(pos_arr) > changes[-1]:
        durations.append(len(pos_arr) - changes[-1])

    avg_duration = float(np.mean([d for d in durations if d > 0])) if durations else 0.0

    pos_steps = np.sum(pos_arr != 0)
    position_frac = pos_steps / max(n_steps, 1)

    action_dist = {
        name: round(actions.count(k) / max(n_steps, 1) * 100, 2)
        for k, name in ACTION_NAMES.items()
    }

    correction_frac = n_corrections / max(n_steps, 1)

    return {
        "total_return": round(total_return, 2),
        "sharpe": round(sharpe, 4),
        "max_drawdown": round(max_dd, 2),
        "buy_and_hold": round(bah, 2),
        "alpha_vs_bah": round(alpha, 2),
        "n_trades": int(n_trades),
        "trade_freq_per_week": round(trade_freq, 2),
        "win_rate_pct": round(win_rate, 2),
        "avg_hold_duration_hours": round(avg_duration, 1),
        "fees_paid_usd": round(fees_paid, 2),
        "position_frac": round(position_frac, 4),
        "action_dist_pct": action_dist,
        "n_corrections": int(n_corrections),
        "correction_frac": round(correction_frac, 4),
        "n_steps": n_steps,
    }

## FUNCIONS DE VISUALITZACIÓ


In [ ]:
def plot_rolling_boxplots(episodes: List[Dict], window_name: str, output_dir: Path):
    """Genera boxplots de mètriques rolling (return, sharpe, maxDD)."""

    returns = np.array([ep["metrics"]["total_return"] for ep in episodes])
    sharpes = np.array([ep["metrics"]["sharpe"] for ep in episodes])
    maxdds = np.array([ep["metrics"]["max_drawdown"] for ep in episodes])

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))

    for ax, vals, label, color in zip(
        axes,
        [returns, sharpes, maxdds],
        ["Retorn total (%)", "Sharpe ratio", "Max Drawdown (%)"],
        ["#2ca02c", "#1f77b4", "#d62728"],
    ):
        bp = ax.boxplot(vals, patch_artist=True,
                        boxprops=dict(facecolor=color, alpha=0.5),
                        medianprops=dict(color="black", linewidth=2),
                        whiskerprops=dict(linewidth=1.5),
                        capprops=dict(linewidth=1.5))
        ax.scatter([1] * len(vals), vals, color=color, alpha=0.6, s=25, zorder=5)
        mu, sd = vals.mean(), vals.std()
        ax.set_title(label, fontsize=11)
        ax.set_xticks([])
        ax.set_xlabel(f"mean={mu:+.3f}  std={sd:.3f}", fontsize=9)
        ax.grid(axis="y", alpha=0.3)
        ax.axhline(0, color="gray", lw=0.8, ls="--")

    plt.suptitle(f"Rolling {window_name.upper()} — {AGENT_LABEL}  ({len(episodes)} episodis)\n{REWARD_TYPE}",
                 fontsize=12, y=1.02)
    plt.tight_layout()

    out_path = output_dir / f"rolling_{window_name}_boxplots.png"
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"    └─ Desat: {out_path.name}")


def plot_rolling_timeseries(episodes: List[Dict], window_name: str, output_dir: Path, timestamps: List):
    """Gràfic sèrie temporal: retorn per episodi."""

    returns = np.array([ep["metrics"]["total_return"] for ep in episodes])
    bahs    = np.array([ep["metrics"]["buy_and_hold"]  for ep in episodes])
    start_dates = [ep["start_date"][:10] for ep in episodes]

    fig, ax = plt.subplots(figsize=(14, 4))

    # B&H bars (fons)
    ax.bar(range(len(bahs)), bahs, color="gray", alpha=0.35, width=0.85,
           label=f"B&H  (mean={bahs.mean():+.2f}%)")

    # Agent bars (primer pla)
    colors = ["#2ca02c" if r >= 0 else "#d62728" for r in returns]
    ax.bar(range(len(returns)), returns, color=colors, alpha=0.80, width=0.55)

    ax.axhline(0, color="black", lw=0.8)
    ax.axhline(returns.mean(), color="navy", lw=1.5, ls="--",
               label=f"Agent Mean: {returns.mean():+.2f}%")
    ax.axhline(bahs.mean(), color="gray", lw=1.2, ls=":",
               label=f"B&H Mean: {bahs.mean():+.2f}%")

    # Etiquetes eix X
    n_labels = min(12, len(start_dates))
    step_lbl = max(1, len(start_dates) // n_labels)
    ax.set_xticks(range(0, len(start_dates), step_lbl))
    ax.set_xticklabels([start_dates[i] for i in range(0, len(start_dates), step_lbl)],
                       rotation=35, ha="right", fontsize=8)

    ax.set_title(f"Retorn per episodi — Rolling {window_name.upper()} — {AGENT_LABEL}\n{REWARD_TYPE}", fontsize=11)
    ax.set_ylabel("Retorn total (%)")
    ax.set_xlabel("Data inici episodi")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    out_path = output_dir / f"rolling_{window_name}_timeseries.png"
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"    └─ Desat: {out_path.name}")


def plot_9_regimes(episodes: List[Dict], window_name: str, output_dir: Path):
    """Genera gràfica 3×3 dels 9 règims de mercat (tendència × volatilitat)."""

    # Calcular tendència (B&H return) i volatilitat per episodi
    price_rets = np.array([
        (ep["prices"][-1] / ep["prices"][0] - 1) * 100
        for ep in episodes
    ])

    log_rets_std = np.array([
        np.std(np.diff(np.log(np.maximum(ep["prices"], 1e-8))))
        for ep in episodes
    ])
    vols_ann = log_rets_std * np.sqrt(STEPS_PER_YEAR) * 100

    # Tercils
    t33, t67 = np.percentile(price_rets, [33.3, 66.7])
    v33, v67 = np.percentile(vols_ann, [33.3, 66.7])

    def trend_bin(r): return 0 if r < t33 else (1 if r < t67 else 2)
    def vol_bin(v): return 0 if v < v33 else (1 if v < v67 else 2)

    # Assignar episodis a cel·les
    cell_members = {}
    for i in range(len(episodes)):
        key = (trend_bin(price_rets[i]), vol_bin(vols_ann[i]))
        cell_members.setdefault(key, []).append(i)

    # Normalització per selecció representant
    t_range = price_rets.max() - price_rets.min()
    v_range = vols_ann.max() - vols_ann.min()
    t_norm = (price_rets - price_rets.min()) / (t_range + 1e-8)
    v_norm = (vols_ann - vols_ann.min()) / (v_range + 1e-8)

    T_TGT = {0: 0.0, 1: 0.5, 2: 1.0}
    V_TGT = {0: 0.0, 1: 0.5, 2: 1.0}

    def best_repr(tb, vb):
        members = cell_members.get((tb, vb), [])
        pool = members if members else list(range(len(episodes)))
        tt, vt = T_TGT[tb], V_TGT[vb]
        dists = [np.hypot(t_norm[i] - tt, v_norm[i] - vt) for i in pool]
        return pool[int(np.argmin(dists))]

    GRID = {(tb, vb): best_repr(tb, vb) for tb in range(3) for vb in range(3)}

    # Colors
    TREND_COLORS = {0: "#8b0000", 1: "#444444", 2: "#006400"}
    TREND_NAMES = {0: "Baixista", 1: "Lateral", 2: "Alcista"}
    VOL_NAMES = {2: "Alta vol.", 1: "Vol. mitj.", 0: "Baixa vol."}

    # Figura 3×3
    fig, axes = plt.subplots(3, 3, figsize=(18, 13),
                             gridspec_kw={"hspace": 0.48, "wspace": 0.28})

    for row_i, vb in enumerate([2, 1, 0]):
        for col_i, tb in enumerate(range(3)):
            ax = axes[row_i, col_i]
            idx = GRID[(tb, vb)]
            ep = episodes[idx]
            px = np.array(ep["prices"])
            acts = ep["actions"]
            pos_seq = _reconstruct_positions(acts)
            m = ep["metrics"]

            # Fons posició
            pos_arr = np.array(pos_seq)
            i = 0
            while i < len(pos_arr):
                p = pos_arr[i]
                j = i + 1
                while j < len(pos_arr) and pos_arr[j] == p:
                    j += 1
                if p == 1:
                    ax.axvspan(i, j - 1, alpha=0.13, color="#2ca02c", lw=0)
                elif p == -1:
                    ax.axvspan(i, j - 1, alpha=0.13, color="#d62728", lw=0)
                i = j

            # Línia preus
            ax.plot(px, color="#1f77b4", lw=0.85, alpha=0.9, zorder=3)

            # Marcadors canvis acció
            changes = [k for k in range(len(acts)) if k == 0 or acts[k] != acts[k - 1]]
            for k in changes:
                a = acts[k]
                if a == 1:
                    ax.scatter(k, px[k], marker="^", color="#2ca02c", s=55,
                               zorder=5, edgecolors="white", linewidths=0.4)
                elif a == 2:
                    ax.scatter(k, px[k], marker="v", color="#d62728", s=55,
                               zorder=5, edgecolors="white", linewidths=0.4)
                elif a == 3:
                    ax.scatter(k, px[k], marker="s", color="#ff7f0e", s=40,
                               zorder=5, edgecolors="white", linewidths=0.4)

            # Títol
            sign_ag = "+" if m["total_return"] >= 0 else ""
            sign_bah = "+" if price_rets[idx] >= 0 else ""
            profit = ep["equities"][-1] - ep["equities"][0]
            sign_p = "+" if profit >= 0 else ""
            avg_dur = m["n_steps"] / m["n_trades"] if m["n_trades"] > 0 else 0

            ax.set_title(
                f"[{TREND_NAMES[tb]} · {VOL_NAMES[vb]}]  Ep {idx+1}\n"
                f"{ep['start_date'][:10]}   "
                f"B&H: {sign_bah}{price_rets[idx]:.1f}%   "
                f"Agent: {sign_ag}{m['total_return']:.1f}%  ({sign_p}${profit:.0f})\n"
                f"Trades: {m['n_trades']}   Avg dur: {avg_dur:.0f}h   Fees: ${m['fees_paid_usd']:.0f}",
                fontsize=7.8, pad=3, color=TREND_COLORS[tb],
            )
            ax.set_xlabel("Step", fontsize=6.5)
            ax.set_ylabel("Preu USD", fontsize=6.5)
            ax.tick_params(labelsize=6)
            ax.grid(alpha=0.22, lw=0.5)
            ax.set_xlim(0, len(px) - 1)

    # Etiquetes columnes
    for col_i, (tb, label) in enumerate([(0, "BAIXISTA"), (1, "LATERAL"), (2, "ALCISTA")]):
        axes[0, col_i].annotate(
            label, xy=(0.5, 1.28), xycoords="axes fraction",
            ha="center", va="bottom", fontsize=10, fontweight="bold",
            color=TREND_COLORS[tb],
        )

    # Etiquetes files
    for row_i, (vb, label) in enumerate([(2, "ALTA VOL."), (1, "VOL. MITJ."), (0, "BAIXA VOL.")]):
        axes[row_i, 0].annotate(
            label, xy=(-0.28, 0.5), xycoords="axes fraction",
            ha="center", va="center", fontsize=9, fontweight="bold",
            color="#333333", rotation=90,
        )

    # Llegenda
    legend_handles = [
        mpatches.Patch(color="#2ca02c", alpha=0.3, label="Posició LONG"),
        mpatches.Patch(color="#d62728", alpha=0.3, label="Posició SHORT"),
        plt.Line2D([0], [0], marker="^", color="w", markerfacecolor="#2ca02c",
                   markersize=8, label="LONG (entrada)"),
        plt.Line2D([0], [0], marker="v", color="w", markerfacecolor="#d62728",
                   markersize=8, label="SHORT (entrada)"),
        plt.Line2D([0], [0], marker="s", color="w", markerfacecolor="#ff7f0e",
                   markersize=7, label="CLOSE (sortida)"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=5,
               fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.015))

    fig.suptitle(
        f"9 Règims de Mercat — {AGENT_LABEL} ({window_name.upper()})  {REWARD_TYPE}\n"
        f"llindars: t=[{t33:+.1f}%, {t67:+.1f}%]  v=[{v33:.0f}%, {v67:.0f}%]",
        fontsize=10, y=1.01,
    )

    out_path = output_dir / f"rolling_{window_name}_9regimes.png"
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"    └─ Desat: {out_path.name}")


def plot_full_period_equity(result: Dict, output_dir: Path):
    """Equity curve vs Buy & Hold per full period."""

    eq_arr = np.array(result["equities"], dtype=np.float64)
    px_arr = np.array(result["prices"], dtype=np.float64)
    m = result["metrics"]

    bah_curve = INITIAL_BALANCE * (px_arr / px_arr[0])

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(eq_arr, color="#1f77b4", lw=1.0, alpha=0.9, label=f"{AGENT_LABEL}")
    ax.plot(np.concatenate([[INITIAL_BALANCE], bah_curve]),
            color="gray", lw=1.0, alpha=0.7, ls="--", label="Buy & Hold")
    ax.axhline(INITIAL_BALANCE, color="black", lw=0.5, ls=":")

    ax.set_title(
        f"Equity Curve — {AGENT_LABEL}  {REWARD_TYPE}\n"
        f"Ret: {m['total_return']:+.2f}%  B&H: {m['buy_and_hold']:+.2f}%  "
        f"Alpha: {m['alpha_vs_bah']:+.2f}%  Sharpe: {m['sharpe']:.4f}",
        fontsize=11
    )
    ax.set_xlabel("Pas")
    ax.set_ylabel("Equity (USD)")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()

    out_path = output_dir / "full_equity.png"
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"    └─ Desat: {out_path.name}")


def plot_full_period_drawdown(result: Dict, output_dir: Path):
    """Drawdown curve per full period."""

    eq_arr = np.array(result["equities"], dtype=np.float64)
    m = result["metrics"]

    running_max = np.maximum.accumulate(eq_arr)
    dd_curve = (eq_arr - running_max) / (running_max + 1e-10) * 100

    fig, ax = plt.subplots(figsize=(16, 4))
    ax.fill_between(range(len(dd_curve)), dd_curve, 0, color="#d62728", alpha=0.4)
    ax.plot(dd_curve, color="#d62728", lw=0.8)
    ax.axhline(0, color="black", lw=0.5)

    ax.set_title(f"Drawdown — {AGENT_LABEL}  (MaxDD: {m['max_drawdown']:.2f}%)  {REWARD_TYPE}", fontsize=11)
    ax.set_xlabel("Pas")
    ax.set_ylabel("Drawdown (%)")
    ax.grid(alpha=0.3)
    plt.tight_layout()

    out_path = output_dir / "full_drawdown.png"
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"    └─ Desat: {out_path.name}")

## CÀRREGA AGENT I MONDAY_STARTS


In [ ]:
print(f"\n{'='*70}")
print(f"  CARREGANT AGENT: {AGENT_TYPE}")
print(f"{'='*70}\n")

model, IS_META_WRAPPER = _load_agent_with_meta(
    agent_path=agent_cfg["path"],
    agent_type=agent_cfg["type"],
    meta_adjustment=META_ADJUSTMENT,
    device=DEVICE,
    loader=agent_cfg.get("loader", "ActorCriticPolicy"),
    best_params_path=agent_cfg.get("best_params_path"),
)

print(f"\n{'='*70}")
print(f"  AGENT CARREGAT ✓")
print(f"{'='*70}\n")

# Monday starts
timestamps = df_test["timestamp"].to_list()

WINDOW_CONFIGS = {
    "4w": 4 * STEPS_PER_WEEK,
    "8w": 8 * STEPS_PER_WEEK,
    "12w": 12 * STEPS_PER_WEEK,
}

monday_starts = {}
for window_name, window_steps in WINDOW_CONFIGS.items():
    starts = [
        i for i, ts in enumerate(timestamps)
        if ts.weekday() == 0 and ts.hour == 0 and ts.minute == 0
        and i + window_steps <= len(df_test)
    ]
    monday_starts[window_name] = starts

print(f"\n{'='*70}")
print(f"  ROLLING WINDOWS — MONDAY STARTS")
print(f"{'='*70}")
for wname in ["4w", "8w", "12w"]:
    starts = monday_starts[wname]
    steps = WINDOW_CONFIGS[wname]
    print(f"\n  {wname.upper()} ({steps} steps):")
    print(f"    Episodes: {len(starts)}")
    if starts:
        print(f"    Primer:   {timestamps[starts[0]]}")

print(f"\n{'='*70}\n")

## EXPERIMENT 1: ROLLING WINDOWS ALL-IN (4w, 8w, 12w)


In [ ]:
def run_rolling_window_allin(
    model,
    window_name: str,
    window_steps: int,
    starts: List[int],
    df_test: pl.DataFrame,
    timestamps: List,
) -> Dict[str, Any]:
    """
    Executa rolling window all-in per una finestra determinada.

    Returns:
        Dict amb clau 'episodes' (llista de dicts amb metrics, equities, etc.)
    """
    env = make_env(df_test, max_episode_steps=window_steps)
    episodes = []

    for ep_idx, start_idx in enumerate(tqdm(starts, desc=f"Ex.1 {window_name}")):
        # Meta-adaptation si aplica
        if IS_META_WRAPPER and META_ADJUSTMENT is not None:
            n_support = META_ADAPT_WEEKS * STEPS_PER_WEEK

            if start_idx >= n_support:
                df_support = df_test[start_idx - n_support : start_idx]
            else:
                need_from_val = n_support - start_idx
                val_part = df_val[-need_from_val:] if need_from_val <= len(df_val) else df_val
                df_support = pl.concat([val_part, df_test[:start_idx]]) if start_idx > 0 else val_part

            adapt_result = model.adapt(
                df_support=df_support,
                mode=META_ADJUSTMENT,
                n_steps=META_ADAPT_STEPS,
                loss_threshold=META_ADAPT_LOSS_THRESHOLD,
                max_steps=META_ADAPT_MAX_STEPS,
            )
        else:
            adapt_result = None

        # Reset episodi
        obs, info = env.reset(options={"episode_start": start_idx})
        if hasattr(model, 'reset_episode'):
            model.reset_episode()

        equities = [info["equity"]]
        actions = []
        prices = []
        n_corrections = 0
        done = False

        # Rollout
        while not done:
            raw_action, _ = model.predict(obs, deterministic=True)
            action = _correct_action(int(raw_action), info["pos_side"])

            if action != int(raw_action):
                n_corrections += 1

            obs, reward, term, trunc, info = env.step(action)
            done = term or trunc

            equities.append(info["equity"])
            actions.append(action)
            prices.append(float(info["current_price"]))

        # Metrics
        metrics = _calculate_metrics(
            equities=equities,
            actions=actions,
            prices=prices,
            n_trades=info["n_trades"],
            fees_paid=info["cum_fees"],
            n_corrections=n_corrections,
        )

        # Guardar episodi
        episode_data = {
            "episode_idx": ep_idx,
            "start_idx": start_idx,
            "start_date": str(timestamps[start_idx]),
            "window_steps": window_steps,
            "equities": [round(e, 4) for e in equities],
            "actions": actions,
            "prices": [round(p, 2) for p in prices],
            "metrics": metrics,
        }

        if adapt_result:
            episode_data["adaptation"] = adapt_result

        episodes.append(episode_data)

    return {"episodes": episodes}


print(f"\n{'#'*70}")
print(f"  EXPERIMENT 1: ROLLING WINDOWS ALL-IN")
print(f"{'#'*70}\n")

ex1_results = {}

for wname in ["4w", "8w", "12w"]:
    print(f"\n{'─'*70}")
    print(f"  Ex.1{wname[0].upper()}: Rolling {wname.upper()}")
    print(f"{'─'*70}\n")

    result = run_rolling_window_allin(
        model=model,
        window_name=wname,
        window_steps=WINDOW_CONFIGS[wname],
        starts=monday_starts[wname],
        df_test=df_test,
        timestamps=timestamps,
    )

    ex1_results[wname] = result

    # Calcular estadístiques agregades
    episodes = result["episodes"]
    if episodes:
        returns = [ep["metrics"]["total_return"] for ep in episodes]
        sharpes = [ep["metrics"]["sharpe"] for ep in episodes]

        print(f"\n  Resultats {wname.upper()}:")
        print(f"    Episodes:       {len(episodes)}")
        print(f"    Retorn mig:     {np.mean(returns):+.2f}% (std={np.std(returns):.2f}%)")
        print(f"    Sharpe mig:     {np.mean(sharpes):+.4f}")
        print(f"    Win rate mig:   {np.mean([ep['metrics']['win_rate_pct'] for ep in episodes]):.1f}%")
        print(f"    Trades mig:     {np.mean([ep['metrics']['n_trades'] for ep in episodes]):.1f}")
        print(f"    B&H mig:        {np.mean([ep['metrics']['buy_and_hold'] for ep in episodes]):+.2f}%")
        print(f"    Alpha mig:      {np.mean([ep['metrics']['alpha_vs_bah'] for ep in episodes]):+.2f}%")

    # Guardar JSON
    out_path = RESULTS_DIR / f"ex1_rolling_allin/ex1{wname[0]}_rolling_{wname}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({
            "agent": AGENT_NAME,
            "exercise": f"1{wname[0]}",
            "window": wname,
            **result
        }, f, indent=2, ensure_ascii=False)

    print(f"\n    Desat: {out_path.name}")

    # Visualitzacions
    if episodes:
        print(f"\n    Generant visualitzacions...")
        plot_rolling_boxplots(episodes, wname, RESULTS_DIR / "ex1_rolling_allin")
        plot_rolling_timeseries(episodes, wname, RESULTS_DIR / "ex1_rolling_allin", timestamps)
        plot_9_regimes(episodes, wname, RESULTS_DIR / "ex1_rolling_allin")

print(f"\n{'#'*70}")
print(f"  EXPERIMENT 1 COMPLETAT ✓")
print(f"{'#'*70}\n")

## EXPERIMENT 2: ROLLING WINDOWS POSITION SIZING (4w, 8w, 12w)


In [ ]:
def run_rolling_window_sizing(
    model,
    window_name: str,
    window_steps: int,
    starts: List[int],
    df_test: pl.DataFrame,
    timestamps: List,
    initial_balance: float = 10_000.0,
    max_position_usd: float = 1_000.0,
) -> Dict[str, Any]:
    """
    Executa rolling window amb position sizing.

    Position sizing:
      - Pressupost inicial: 10,000 USD
      - Màxim per trade: 1,000 USD (10%)
      - Només 1 posició simultània
      - Tancaments: 100% posició
    """
    env = make_env(df_test, max_episode_steps=window_steps)
    episodes = []

    for ep_idx, start_idx in enumerate(tqdm(starts, desc=f"Ex.2 {window_name}")):
        # Meta-adaptation (igual que Ex.1)
        if IS_META_WRAPPER and META_ADJUSTMENT is not None:
            n_support = META_ADAPT_WEEKS * STEPS_PER_WEEK

            if start_idx >= n_support:
                df_support = df_test[start_idx - n_support : start_idx]
            else:
                need_from_val = n_support - start_idx
                val_part = df_val[-need_from_val:] if need_from_val <= len(df_val) else df_val
                df_support = pl.concat([val_part, df_test[:start_idx]]) if start_idx > 0 else val_part

            adapt_result = model.adapt(
                df_support=df_support,
                mode=META_ADJUSTMENT,
                n_steps=META_ADAPT_STEPS,
                loss_threshold=META_ADAPT_LOSS_THRESHOLD,
                max_steps=META_ADAPT_MAX_STEPS,
            )
        else:
            adapt_result = None

        # Reset
        obs, info = env.reset(options={"episode_start": start_idx})
        if hasattr(model, 'reset_episode'):
            model.reset_episode()

        # Tracking position sizing
        balance = initial_balance
        position_open = False
        position_entry_price = 0.0
        position_size_usd = 0.0
        position_side = 0  # 0=FLAT, 1=LONG, -1=SHORT

        equities_sized = [balance]
        actions = []
        prices = []
        n_corrections = 0
        n_trades_sized = 0
        fees_sized = 0.0
        done = False

        # Rollout amb position sizing
        while not done:
            raw_action, _ = model.predict(obs, deterministic=True)
            action = _correct_action(int(raw_action), info["pos_side"])

            if action != int(raw_action):
                n_corrections += 1

            current_price = float(info["current_price"])

            # Lògica position sizing
            if action == 1 and not position_open:  # LONG
                position_open = True
                position_side = 1
                position_entry_price = current_price
                position_size_usd = min(max_position_usd, balance)
                n_trades_sized += 1
                fees_sized += position_size_usd * 0.0016  # taker fee Kraken

            elif action == 2 and not position_open:  # SHORT
                position_open = True
                position_side = -1
                position_entry_price = current_price
                position_size_usd = min(max_position_usd, balance)
                n_trades_sized += 1
                fees_sized += position_size_usd * 0.0016

            elif action == 3 and position_open:  # CLOSE
                price_change = (current_price - position_entry_price) / position_entry_price
                if position_side == -1:
                    price_change = -price_change

                pnl = position_size_usd * price_change
                balance += pnl
                fees_sized += abs(pnl) * 0.0016

                position_open = False
                position_side = 0
                position_size_usd = 0.0

            # Actualitzar equity sized
            if position_open:
                price_change = (current_price - position_entry_price) / position_entry_price
                if position_side == -1:
                    price_change = -price_change
                unrealized_pnl = position_size_usd * price_change
                equity_sized = balance + unrealized_pnl
            else:
                equity_sized = balance

            equities_sized.append(equity_sized)
            actions.append(action)
            prices.append(current_price)

            # Step entorn
            obs, reward, term, trunc, info = env.step(action)
            done = term or trunc

        # Tancar posició si queda oberta al final
        if position_open:
            current_price = prices[-1]
            price_change = (current_price - position_entry_price) / position_entry_price
            if position_side == -1:
                price_change = -price_change
            pnl = position_size_usd * price_change
            balance += pnl
            equities_sized[-1] = balance

        # Metrics amb equity sized
        metrics = _calculate_metrics(
            equities=equities_sized,
            actions=actions,
            prices=prices,
            n_trades=n_trades_sized,
            fees_paid=fees_sized,
            n_corrections=n_corrections,
        )

        # Guardar episodi
        episode_data = {
            "episode_idx": ep_idx,
            "start_idx": start_idx,
            "start_date": str(timestamps[start_idx]),
            "window_steps": window_steps,
            "initial_balance": initial_balance,
            "max_position_usd": max_position_usd,
            "equities": [round(e, 4) for e in equities_sized],
            "actions": actions,
            "prices": [round(p, 2) for p in prices],
            "metrics": metrics,
        }

        if adapt_result:
            episode_data["adaptation"] = adapt_result

        episodes.append(episode_data)

    return {"episodes": episodes}


print(f"\n{'#'*70}")
print(f"  EXPERIMENT 2: ROLLING WINDOWS POSITION SIZING")
print(f"{'#'*70}\n")

ex2_results = {}

for wname in ["4w", "8w", "12w"]:
    print(f"\n{'─'*70}")
    print(f"  Ex.2{wname[0].upper()}: Rolling {wname.upper()} (Position Sizing)")
    print(f"{'─'*70}\n")

    result = run_rolling_window_sizing(
        model=model,
        window_name=wname,
        window_steps=WINDOW_CONFIGS[wname],
        starts=monday_starts[wname],
        df_test=df_test,
        timestamps=timestamps,
        initial_balance=10_000.0,
        max_position_usd=1_000.0,
    )

    ex2_results[wname] = result

    # Estadístiques
    episodes = result["episodes"]
    if episodes:
        returns = [ep["metrics"]["total_return"] for ep in episodes]
        sharpes = [ep["metrics"]["sharpe"] for ep in episodes]

        print(f"\n  Resultats {wname.upper()} (Sizing):")
        print(f"    Episodes:       {len(episodes)}")
        print(f"    Retorn mig:     {np.mean(returns):+.2f}% (std={np.std(returns):.2f}%)")
        print(f"    Sharpe mig:     {np.mean(sharpes):+.4f}")
        print(f"    Win rate mig:   {np.mean([ep['metrics']['win_rate_pct'] for ep in episodes]):.1f}%")
        print(f"    Trades mig:     {np.mean([ep['metrics']['n_trades'] for ep in episodes]):.1f}")
        print(f"    B&H mig:        {np.mean([ep['metrics']['buy_and_hold'] for ep in episodes]):+.2f}%")
        print(f"    Alpha mig:      {np.mean([ep['metrics']['alpha_vs_bah'] for ep in episodes]):+.2f}%")

    # Guardar JSON
    out_path = RESULTS_DIR / f"ex2_rolling_sizing/ex2{wname[0]}_rolling_{wname}_sizing.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({
            "agent": AGENT_NAME,
            "exercise": f"2{wname[0]}",
            "window": wname,
            "sizing": {"initial_balance": 10_000.0, "max_position_usd": 1_000.0},
            **result
        }, f, indent=2, ensure_ascii=False)

    print(f"\n    Desat: {out_path.name}")

    # Visualitzacions
    if episodes:
        print(f"\n    Generant visualitzacions...")
        plot_rolling_boxplots(episodes, f"{wname}_sizing", RESULTS_DIR / "ex2_rolling_sizing")
        plot_rolling_timeseries(episodes, f"{wname}_sizing", RESULTS_DIR / "ex2_rolling_sizing", timestamps)
        plot_9_regimes(episodes, f"{wname}_sizing", RESULTS_DIR / "ex2_rolling_sizing")

print(f"\n{'#'*70}")
print(f"  EXPERIMENT 2 COMPLETAT ✓")
print(f"{'#'*70}\n")

## EXPERIMENT 3: FULL PERIOD ALL-IN


In [ ]:
def run_full_period_allin(
    model,
    df_test: pl.DataFrame,
    timestamps: List,
    rebalance_weekly: bool = False,
) -> Dict[str, Any]:
    """
    Executa full period all-in sobre tot el split test.

    Args:
        rebalance_weekly: Si True (agents meta amb adjustment), re-adapta cada setmana
    """
    env = make_env(df_test, max_episode_steps=len(df_test)-1, dd_threshold=1.0)

    # Adaptació inicial si meta
    if IS_META_WRAPPER and META_ADJUSTMENT is not None:
        print("  Adaptació inicial sobre df_val...")
        n_support = META_ADAPT_WEEKS * STEPS_PER_WEEK
        df_support = df_val[-n_support:] if len(df_val) >= n_support else df_val

        adapt_result_init = model.adapt(
            df_support=df_support,
            mode=META_ADJUSTMENT,
            n_steps=META_ADAPT_STEPS,
            loss_threshold=META_ADAPT_LOSS_THRESHOLD,
            max_steps=META_ADAPT_MAX_STEPS,
        )
        adaptation_log = [{"step": 0, **adapt_result_init}]
    else:
        adaptation_log = []

    # Reset
    obs, info = env.reset(options={"episode_start": 0})
    if hasattr(model, 'reset_episode'):
        model.reset_episode()

    equities = [info["equity"]]
    actions = []
    prices = []
    n_corrections = 0
    step_count = 0
    done = False

    # Rollout
    pbar = tqdm(total=len(df_test), desc="Ex.3 Full Period")

    while not done:
        # Rebalance setmanal si aplica
        if rebalance_weekly and IS_META_WRAPPER and META_ADJUSTMENT is not None:
            if step_count > 0 and step_count % STEPS_PER_WEEK == 0:
                print(f"\n  Rebalance setmanal (step {step_count})...")
                n_support = META_ADAPT_WEEKS * STEPS_PER_WEEK
                support_start = max(0, step_count - n_support)
                df_support = df_test[support_start:step_count]

                adapt_result = model.adapt(
                    df_support=df_support,
                    mode=META_ADJUSTMENT,
                    n_steps=META_ADAPT_STEPS,
                    loss_threshold=META_ADAPT_LOSS_THRESHOLD,
                    max_steps=META_ADAPT_MAX_STEPS,
                )
                adaptation_log.append({"step": step_count, **adapt_result})

        raw_action, _ = model.predict(obs, deterministic=True)
        action = _correct_action(int(raw_action), info["pos_side"])

        if action != int(raw_action):
            n_corrections += 1

        obs, reward, term, trunc, info = env.step(action)
        done = term or trunc

        equities.append(info["equity"])
        actions.append(action)
        prices.append(float(info["current_price"]))

        step_count += 1
        pbar.update(1)

    pbar.close()

    # Metrics
    metrics = _calculate_metrics(
        equities=equities,
        actions=actions,
        prices=prices,
        n_trades=info["n_trades"],
        fees_paid=info["cum_fees"],
        n_corrections=n_corrections,
    )

    return {
        "n_steps": step_count,
        "rebalance_weekly": rebalance_weekly,
        "adaptation_log": adaptation_log,
        "equities": [round(e, 4) for e in equities],
        "actions": actions,
        "prices": [round(p, 2) for p in prices],
        "metrics": metrics,
    }


print(f"\n{'#'*70}")
print(f"  EXPERIMENT 3: FULL PERIOD ALL-IN")
print(f"{'#'*70}\n")

rebalance = IS_META_WRAPPER and META_ADJUSTMENT is not None

ex3_result = run_full_period_allin(
    model=model,
    df_test=df_test,
    timestamps=timestamps,
    rebalance_weekly=rebalance,
)

print(f"\n  Resultats Full Period All-in:")
print(f"    Steps:          {ex3_result['n_steps']:,}")
print(f"    Retorn total:   {ex3_result['metrics']['total_return']:+.2f}%")
print(f"    Buy & Hold:     {ex3_result['metrics']['buy_and_hold']:+.2f}%")
print(f"    Alpha:          {ex3_result['metrics']['alpha_vs_bah']:+.2f}%")
print(f"    Sharpe:         {ex3_result['metrics']['sharpe']:+.4f}")
print(f"    Max DD:         {ex3_result['metrics']['max_drawdown']:.2f}%")
print(f"    Trades:         {ex3_result['metrics']['n_trades']}")
if rebalance:
    print(f"    Rebalances:     {len(ex3_result['adaptation_log'])}")

# Guardar
out_path = RESULTS_DIR / "ex3_full_allin/ex3_full_period_allin.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump({
        "agent": AGENT_NAME,
        "exercise": "3",
        **ex3_result
    }, f, indent=2, ensure_ascii=False)

print(f"\n  Desat: {out_path.name}")

# Visualitzacions Ex.3
print(f"\n  Generant visualitzacions...")
plot_full_period_equity(ex3_result, RESULTS_DIR / "ex3_full_allin")
plot_full_period_drawdown(ex3_result, RESULTS_DIR / "ex3_full_allin")

print(f"\n{'#'*70}")
print(f"  EXPERIMENT 3 COMPLETAT ✓")
print(f"{'#'*70}\n")

## EXPERIMENT 4: FULL PERIOD POSITION SIZING


In [ ]:
def run_full_period_sizing(
    model,
    df_test: pl.DataFrame,
    timestamps: List,
    initial_balance: float = 10_000.0,
    max_position_usd: float = 1_000.0,
    rebalance_weekly: bool = False,
) -> Dict[str, Any]:
    """
    Executa full period amb position sizing sobre tot el split test.
    """
    env = make_env(df_test, max_episode_steps=len(df_test)-1, dd_threshold=1.0)

    # Adaptació inicial si meta
    if IS_META_WRAPPER and META_ADJUSTMENT is not None:
        print("  Adaptació inicial sobre df_val...")
        n_support = META_ADAPT_WEEKS * STEPS_PER_WEEK
        df_support = df_val[-n_support:] if len(df_val) >= n_support else df_val

        adapt_result_init = model.adapt(
            df_support=df_support,
            mode=META_ADJUSTMENT,
            n_steps=META_ADAPT_STEPS,
            loss_threshold=META_ADAPT_LOSS_THRESHOLD,
            max_steps=META_ADAPT_MAX_STEPS,
        )
        adaptation_log = [{"step": 0, **adapt_result_init}]
    else:
        adaptation_log = []

    # Reset
    obs, info = env.reset(options={"episode_start": 0})
    if hasattr(model, 'reset_episode'):
        model.reset_episode()

    # Position sizing tracking
    balance = initial_balance
    position_open = False
    position_entry_price = 0.0
    position_size_usd = 0.0
    position_side = 0

    equities_sized = [balance]
    actions = []
    prices = []
    n_corrections = 0
    n_trades_sized = 0
    fees_sized = 0.0
    step_count = 0
    done = False

    # Rollout
    pbar = tqdm(total=len(df_test), desc="Ex.4 Full Period Sizing")

    while not done:
        # Rebalance setmanal si aplica
        if rebalance_weekly and IS_META_WRAPPER and META_ADJUSTMENT is not None:
            if step_count > 0 and step_count % STEPS_PER_WEEK == 0:
                print(f"\n  Rebalance setmanal (step {step_count})...")
                n_support = META_ADAPT_WEEKS * STEPS_PER_WEEK
                support_start = max(0, step_count - n_support)
                df_support = df_test[support_start:step_count]

                adapt_result = model.adapt(
                    df_support=df_support,
                    mode=META_ADJUSTMENT,
                    n_steps=META_ADAPT_STEPS,
                    loss_threshold=META_ADAPT_LOSS_THRESHOLD,
                    max_steps=META_ADAPT_MAX_STEPS,
                )
                adaptation_log.append({"step": step_count, **adapt_result})

        raw_action, _ = model.predict(obs, deterministic=True)
        action = _correct_action(int(raw_action), info["pos_side"])

        if action != int(raw_action):
            n_corrections += 1

        current_price = float(info["current_price"])

        # Position sizing logic
        if action == 1 and not position_open:  # LONG
            position_open = True
            position_side = 1
            position_entry_price = current_price
            position_size_usd = min(max_position_usd, balance)
            n_trades_sized += 1
            fees_sized += position_size_usd * 0.0016

        elif action == 2 and not position_open:  # SHORT
            position_open = True
            position_side = -1
            position_entry_price = current_price
            position_size_usd = min(max_position_usd, balance)
            n_trades_sized += 1
            fees_sized += position_size_usd * 0.0016

        elif action == 3 and position_open:  # CLOSE
            price_change = (current_price - position_entry_price) / position_entry_price
            if position_side == -1:
                price_change = -price_change

            pnl = position_size_usd * price_change
            balance += pnl
            fees_sized += abs(pnl) * 0.0016

            position_open = False
            position_side = 0
            position_size_usd = 0.0

        # Update equity
        if position_open:
            price_change = (current_price - position_entry_price) / position_entry_price
            if position_side == -1:
                price_change = -price_change
            unrealized_pnl = position_size_usd * price_change
            equity_sized = balance + unrealized_pnl
        else:
            equity_sized = balance

        equities_sized.append(equity_sized)
        actions.append(action)
        prices.append(current_price)

        # Step
        obs, reward, term, trunc, info = env.step(action)
        done = term or trunc

        step_count += 1
        pbar.update(1)

    pbar.close()

    # Tancar posició si queda oberta
    if position_open:
        current_price = prices[-1]
        price_change = (current_price - position_entry_price) / position_entry_price
        if position_side == -1:
            price_change = -price_change
        pnl = position_size_usd * price_change
        balance += pnl
        equities_sized[-1] = balance

    # Metrics
    metrics = _calculate_metrics(
        equities=equities_sized,
        actions=actions,
        prices=prices,
        n_trades=n_trades_sized,
        fees_paid=fees_sized,
        n_corrections=n_corrections,
    )

    return {
        "n_steps": step_count,
        "initial_balance": initial_balance,
        "max_position_usd": max_position_usd,
        "rebalance_weekly": rebalance_weekly,
        "adaptation_log": adaptation_log,
        "equities": [round(e, 4) for e in equities_sized],
        "actions": actions,
        "prices": [round(p, 2) for p in prices],
        "metrics": metrics,
    }


print(f"\n{'#'*70}")
print(f"  EXPERIMENT 4: FULL PERIOD POSITION SIZING")
print(f"{'#'*70}\n")

rebalance = IS_META_WRAPPER and META_ADJUSTMENT is not None

ex4_result = run_full_period_sizing(
    model=model,
    df_test=df_test,
    timestamps=timestamps,
    initial_balance=10_000.0,
    max_position_usd=1_000.0,
    rebalance_weekly=rebalance,
)

print(f"\n  Resultats Full Period Sizing:")
print(f"    Steps:          {ex4_result['n_steps']:,}")
print(f"    Retorn total:   {ex4_result['metrics']['total_return']:+.2f}%")
print(f"    Buy & Hold:     {ex4_result['metrics']['buy_and_hold']:+.2f}%")
print(f"    Alpha:          {ex4_result['metrics']['alpha_vs_bah']:+.2f}%")
print(f"    Sharpe:         {ex4_result['metrics']['sharpe']:+.4f}")
print(f"    Max DD:         {ex4_result['metrics']['max_drawdown']:.2f}%")
print(f"    Trades:         {ex4_result['metrics']['n_trades']}")
if rebalance:
    print(f"    Rebalances:     {len(ex4_result['adaptation_log'])}")

# Guardar
out_path = RESULTS_DIR / "ex4_full_sizing/ex4_full_period_sizing.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump({
        "agent": AGENT_NAME,
        "exercise": "4",
        **ex4_result
    }, f, indent=2, ensure_ascii=False)

print(f"\n  Desat: {out_path.name}")

# Visualitzacions Ex.4
print(f"\n  Generant visualitzacions...")
plot_full_period_equity(ex4_result, RESULTS_DIR / "ex4_full_sizing")
plot_full_period_drawdown(ex4_result, RESULTS_DIR / "ex4_full_sizing")

print(f"\n{'#'*70}")
print(f"  EXPERIMENT 4 COMPLETAT ✓")
print(f"{'#'*70}\n")

## RESUM FINAL COMPARATIU


In [ ]:
print(f"\n{'='*70}")
print(f"  RESUM FINAL COMPARATIU — {AGENT_NAME}")
print(f"{'='*70}\n")

# Recopilar mètriques clau de cada experiment
summary = {
    "agent": AGENT_NAME,
    "agent_type": AGENT_TYPE,
    "meta_adjustment": META_ADJUSTMENT,
    "timestamp": datetime.now().isoformat(),
    "exercises": {}
}

# Ex.1: Agregats rolling all-in
for wname in ["4w", "8w", "12w"]:
    if wname in ex1_results and ex1_results[wname]["episodes"]:
        eps = ex1_results[wname]["episodes"]
        summary["exercises"][f"ex1_{wname}"] = {
            "n_episodes": len(eps),
            "mean_return": float(np.mean([e["metrics"]["total_return"] for e in eps])),
            "mean_sharpe": float(np.mean([e["metrics"]["sharpe"] for e in eps])),
            "mean_max_dd": float(np.mean([e["metrics"]["max_drawdown"] for e in eps])),
            "mean_bah": float(np.mean([e["metrics"]["buy_and_hold"] for e in eps])),
            "mean_alpha": float(np.mean([e["metrics"]["alpha_vs_bah"] for e in eps])),
            "mean_trades": float(np.mean([e["metrics"]["n_trades"] for e in eps])),
        }

# Ex.2: Agregats rolling sizing
for wname in ["4w", "8w", "12w"]:
    if wname in ex2_results and ex2_results[wname]["episodes"]:
        eps = ex2_results[wname]["episodes"]
        summary["exercises"][f"ex2_{wname}"] = {
            "n_episodes": len(eps),
            "mean_return": float(np.mean([e["metrics"]["total_return"] for e in eps])),
            "mean_sharpe": float(np.mean([e["metrics"]["sharpe"] for e in eps])),
            "mean_max_dd": float(np.mean([e["metrics"]["max_drawdown"] for e in eps])),
            "mean_trades": float(np.mean([e["metrics"]["n_trades"] for e in eps])),
        }

# Ex.3: Full period all-in
summary["exercises"]["ex3_full_allin"] = {
    "n_steps": ex3_result["n_steps"],
    "return": ex3_result["metrics"]["total_return"],
    "bah": ex3_result["metrics"]["buy_and_hold"],
    "sharpe": ex3_result["metrics"]["sharpe"],
    "max_dd": ex3_result["metrics"]["max_drawdown"],
    "alpha_vs_bah": ex3_result["metrics"]["alpha_vs_bah"],
    "n_trades": ex3_result["metrics"]["n_trades"],
}

# Ex.4: Full period sizing
summary["exercises"]["ex4_full_sizing"] = {
    "n_steps": ex4_result["n_steps"],
    "return": ex4_result["metrics"]["total_return"],
    "bah": ex4_result["metrics"]["buy_and_hold"],
    "sharpe": ex4_result["metrics"]["sharpe"],
    "max_dd": ex4_result["metrics"]["max_drawdown"],
    "alpha_vs_bah": ex4_result["metrics"]["alpha_vs_bah"],
    "n_trades": ex4_result["metrics"]["n_trades"],
}

# Guardar resum
summary_path = RESULTS_DIR / "summary_all_exercises.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# Print taula comparativa
print("\nTaula Comparativa (retorn mig / Sharpe):\n")
print(f"  {'Experiment':<20} {'Retorn':>10} {'B&H':>10} {'Alpha':>10} {'Sharpe':>10} {'MaxDD':>10} {'Trades':>10}")
print(f"  {'-'*82}")

for ex_name, ex_data in summary["exercises"].items():
    ret = ex_data.get("mean_return") or ex_data.get("return", 0)
    bah = ex_data.get("mean_bah")    or ex_data.get("bah", 0)
    alp = ex_data.get("mean_alpha")  or ex_data.get("alpha_vs_bah", 0)
    shp = ex_data.get("mean_sharpe") or ex_data.get("sharpe", 0)
    mdd = ex_data.get("mean_max_dd") or ex_data.get("max_dd", 0)
    trd = ex_data.get("mean_trades") or ex_data.get("n_trades", 0)

    print(f"  {ex_name:<20} {ret:>+9.2f}% {bah:>+9.2f}% {alp:>+9.2f}% {shp:>10.4f} {mdd:>9.2f}% {trd:>10.1f}")

print(f"\n{'='*70}")
print(f"\nResum desat: {summary_path.name}")

print("\n" + "="*70)
print("  ✓ TOTS ELS EXPERIMENTS COMPLETATS")
print("="*70)
print("\nFitxers generats:")
print(f"  • config.json")
print(f"  • monday_info.json")
print(f"  • Ex.1: 3 JSON (4w, 8w, 12w rolling all-in)")
print(f"  • Ex.2: 3 JSON (4w, 8w, 12w rolling sizing)")
print(f"  • Ex.3: 1 JSON (full period all-in)")
print(f"  • Ex.4: 1 JSON (full period sizing)")
print(f"  • summary_all_exercises.json")
if IS_META_WRAPPER and META_ADJUSTMENT:
    print(f"  • carbon/emissions.csv (CodeCarbon)")

print(f"\nResultats a: {RESULTS_DIR}")
print("="*70)

## MAIN EXECUTION


In [ ]:
if __name__ == "__main__":
    print("\n" + "="*70)
    print("  SCRIPT 10d EVALUATION — DESENVOLUPAMENT")
    print("="*70)
    print("\nConfigurat i llest per executar experiments.")
    print(f"Agent: {AGENT_NAME}")
    print(f"Results: {RESULTS_DIR}")